# Tugas 1 Crawling Detik.com

In [1]:
# Jika belum install library, jalankan sekali saja
!pip install requests beautifulsoup4 trafilatura pandas openpyxl tqdm

  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)

   -------------------------- ------------- 2/3 [openpyxl]
   -------------------------- ------------- 2/3 [openpyxl]
   -------------------------- ------------- 2/3 [openpyxl]
   -------------------------- ------------- 2/3 [openpyxl]
   ---------------------------------------- 3/3 [openpyxl]



Kode tersebut digunakan untuk menginstal beberapa library Python yang dibutuhkan dalam proses crawling dan penambangan web. Perintah pip install digunakan untuk memasang library ke dalam environment Python yang sedang digunakan. Library requests berfungsi untuk mengirim permintaan ke website dan mengambil halaman web, sedangkan beautifulsoup4 digunakan untuk membaca struktur HTML dan mengambil elemen tertentu seperti judul, link, atau informasi lainnya. Library trafilatura digunakan untuk mengekstraksi isi utama artikel dari halaman web agar teks yang diperoleh lebih bersih. Selanjutnya, pandas digunakan untuk mengolah hasil crawling dalam bentuk tabel atau DataFrame, sementara openpyxl digunakan untuk membaca maupun menyimpan data dalam format Excel (.xlsx). Terakhir, tqdm digunakan untuk menampilkan progress bar sehingga proses crawling dalam jumlah banyak dapat dipantau. Perintah instalasi ini cukup dijalankan satu kali selama library tersebut sudah terpasang pada environment Python yang digunakan.

In [2]:
import re
import time
import random
import requests
import pandas as pd
import trafilatura

from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlsplit, urlunsplit
from tqdm import tqdm


# ============================================================
# 1. KONFIGURASI
# ============================================================

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/152.0 Safari/537.36"
    )
}

session = requests.Session()
session.headers.update(HEADERS)


kategori = {
    "sport": {
        "url": "https://sport.detik.com/indeks",
        "domain": "sport.detik.com"
    },

    "finance": {
        "url": "https://finance.detik.com/indeks",
        "domain": "finance.detik.com"
    }
}

Kode tersebut merupakan bagian persiapan awal program crawling data dari website Detik.com, khususnya untuk mengambil data dari kategori Sport dan Finance. Pertama, beberapa library diimpor sesuai kebutuhan, yaitu re untuk melakukan pencarian dan manipulasi pola teks, time dan random untuk mengatur jeda secara acak antarpermintaan agar proses crawling tidak dilakukan terlalu cepat, requests untuk mengakses halaman website, pandas untuk mengolah data hasil crawling, serta trafilatura untuk mengekstraksi isi utama artikel. Selain itu, BeautifulSoup digunakan untuk membaca dan mengolah struktur HTML, sedangkan urljoin, urlsplit, dan urlunsplit digunakan untuk mengolah, menggabungkan, dan menormalisasi URL. tqdm digunakan untuk menampilkan progress bar sehingga perkembangan proses crawling dapat dipantau.

Bagian konfigurasi kemudian menentukan HEADERS yang berisi informasi User-Agent agar request dari program terlihat seperti berasal dari browser biasa. Selanjutnya, requests.Session() digunakan untuk membuat satu sesi koneksi yang dapat digunakan berulang kali selama proses crawling, kemudian session.headers.update(HEADERS) menerapkan User-Agent tersebut pada setiap request. Variabel kategori merupakan dictionary yang digunakan untuk menentukan sumber data yang akan di-crawl. Pada kategori sport, program akan mengambil data dari halaman indeks https://sport.detik.com/indeks dengan domain sport.detik.com, sedangkan kategori finance menggunakan halaman https://finance.detik.com/indeks dengan domain finance.detik.com.

In [3]:
def normalize_url(url):
    """
    Menghapus query parameter dan fragment dari URL.
    """

    bagian = urlsplit(url)

    url_bersih = urlunsplit(
        (
            bagian.scheme,
            bagian.netloc,
            bagian.path,
            "",
            ""
        )
    )

    return url_bersih

Fungsi normalize_url(url) digunakan untuk membersihkan URL sebelum diproses dalam crawling, terutama dengan menghapus query parameter dan fragment yang terdapat pada URL. Pertama, urlsplit(url) memecah URL menjadi beberapa bagian, yaitu skema (https), domain, path, query, dan fragment. Setelah itu, urlunsplit() menyusun kembali URL tersebut dengan mempertahankan bagian skema, domain, dan path, sementara bagian query dan fragment dikosongkan. Hasil akhirnya adalah URL yang lebih bersih dan seragam. Hal ini berguna dalam proses crawling karena satu halaman yang sama terkadang memiliki URL berbeda hanya karena adanya parameter tambahan seperti ?utm_source=... atau fragment seperti #komentar, sehingga pembersihan URL dapat membantu menghindari pengambilan halaman yang sama secara berulang.

In [4]:
def ambil_link_berita(index_url, domain, target=150, max_page=20):

    links = []
    sudah_ada = set()

    for page in range(1, max_page + 1):

        url_page = f"{index_url}?page={page}"

        print(f"Membaca halaman {page}")

        try:

            response = session.get(
                url_page,
                timeout=20
            )

            response.raise_for_status()

        except Exception as e:

            print("Error:", e)
            continue


        soup = BeautifulSoup(
            response.text,
            "html.parser"
        )


        for tag in soup.find_all("a", href=True):

            href = urljoin(
                url_page,
                tag["href"]
            )

            href = normalize_url(href)

            parsed = urlsplit(href)


            # URL artikel Detik biasanya memiliki /d-xxxx
            if (
                parsed.netloc == domain
                and re.search(r"/d-\d+", parsed.path)
            ):

                if href not in sudah_ada:

                    sudah_ada.add(href)

                    links.append(href)


        print(
            "Jumlah link:",
            len(links)
        )


        if len(links) >= target:
            break


        # delay
        time.sleep(
            random.uniform(1, 2)
        )


    return links

Fungsi ambil_link_berita() digunakan untuk mengambil dan mengumpulkan URL artikel berita dari halaman indeks Detik.com secara otomatis hingga mencapai jumlah link yang ditentukan. Fungsi ini menerima index_url sebagai alamat halaman indeks, domain sebagai domain kategori berita, target sebagai jumlah link yang ingin dikumpulkan dengan nilai awal 150, serta max_page sebagai batas maksimal halaman yang akan dibaca yaitu 20 halaman. Program membuat list links untuk menyimpan URL artikel dan sudah_ada berupa set untuk memastikan URL yang sama tidak dikumpulkan berulang kali. Selanjutnya, program melakukan perulangan pada setiap halaman indeks dengan membentuk URL menggunakan ?page={page}, kemudian mengambil halaman tersebut menggunakan session.get() dengan batas waktu 20 detik. Jika terjadi kesalahan saat mengambil halaman, program menampilkan pesan error dan melanjutkan ke halaman berikutnya. Setelah halaman berhasil diperoleh, BeautifulSoup digunakan untuk membaca struktur HTML dan mencari seluruh tag a yang memiliki atribut href. Setiap link kemudian digabungkan dengan URL halaman menggunakan urljoin() dan dibersihkan menggunakan normalize_url(). URL tersebut kemudian diperiksa menggunakan urlsplit() untuk memastikan domainnya sesuai dengan kategori yang sedang diambil dan path URL mengandung pola /d-xxxx, yang merupakan pola URL artikel Detik. Jika URL belum pernah ditemukan, URL tersebut ditambahkan ke sudah_ada dan links. Setelah selesai membaca satu halaman, jumlah link yang telah diperoleh ditampilkan. Jika jumlah link sudah mencapai target, proses crawling dihentikan menggunakan break. Jika belum mencapai target, program memberikan jeda acak antara 1 hingga 2 detik menggunakan random.uniform() sebelum melanjutkan ke halaman berikutnya. Setelah proses selesai, fungsi mengembalikan seluruh URL artikel yang berhasil dikumpulkan melalui return links.

In [5]:
link_sport = ambil_link_berita(
    kategori["sport"]["url"],
    kategori["sport"]["domain"],
    target=150
)

print(
    "Total kandidat Sport:",
    len(link_sport)
)

Membaca halaman 1
Jumlah link: 20
Membaca halaman 2
Jumlah link: 40
Membaca halaman 3
Jumlah link: 60
Membaca halaman 4
Jumlah link: 79
Membaca halaman 5
Jumlah link: 98
Membaca halaman 6
Jumlah link: 117
Membaca halaman 7
Jumlah link: 137
Membaca halaman 8
Jumlah link: 157
Total kandidat Sport: 157


Kode tersebut digunakan untuk memanggil fungsi ambil_link_berita() khusus untuk mengambil link berita dari kategori Sport Detik.com. Pada bagian kategori["sport"]["url"], program mengambil URL halaman indeks Sport yang sebelumnya sudah ditentukan, yaitu https://sport.detik.com/indeks, sedangkan kategori["sport"]["domain"] digunakan untuk menentukan domain yang harus sesuai, yaitu sport.detik.com. Parameter target=150 menunjukkan bahwa program akan berusaha mengumpulkan 150 link artikel Sport. Hasil proses crawling kemudian disimpan ke dalam variabel link_sport. Setelah itu, print("Total kandidat Sport:", len(link_sport)) digunakan untuk menampilkan jumlah link berita Sport yang berhasil dikumpulkan. Jadi, secara sederhana, kode ini memulai proses pengambilan maksimal 150 kandidat link berita Sport dan menampilkan jumlah link yang berhasil diperoleh.

In [7]:
link_finance = ambil_link_berita(
    kategori["finance"]["url"],
    kategori["finance"]["domain"],
    target=150
)

print(
    "Total kandidat Finance:",
    len(link_finance)
)

Membaca halaman 1
Jumlah link: 18
Membaca halaman 2
Jumlah link: 38
Membaca halaman 3
Jumlah link: 57
Membaca halaman 4
Jumlah link: 77
Membaca halaman 5
Jumlah link: 94
Membaca halaman 6
Jumlah link: 114
Membaca halaman 7
Jumlah link: 134
Membaca halaman 8
Jumlah link: 154
Total kandidat Finance: 154


Kode tersebut digunakan untuk memulai proses pengambilan link berita dari kategori Finance Detik.com. Fungsi ambil_link_berita() dipanggil dengan menggunakan URL indeks dan domain Finance yang sebelumnya telah disimpan dalam dictionary kategori. Parameter target=150 menunjukkan bahwa program akan berusaha mengumpulkan 150 link artikel berita Finance. Hasil crawling kemudian disimpan ke dalam variabel link_finance. Setelah proses selesai, print("Total kandidat Finance:", len(link_finance)) digunakan untuk menampilkan jumlah link berita Finance yang berhasil diperoleh. Jadi, kode ini memiliki fungsi yang sama dengan kode sebelumnya untuk kategori Sport, tetapi sumber data yang digunakan adalah kategori Finance, sehingga program akan menghasilkan daftar kandidat link berita Finance untuk tahap crawling berikutnya.

In [8]:
def bersihkan_teks(teks):

    if teks is None:
        return None


    # menghapus teks iklan
    teks = re.sub(
        r"ADVERTISEMENT",
        " ",
        teks,
        flags=re.IGNORECASE
    )


    teks = re.sub(
        r"SCROLL TO CONTINUE WITH CONTENT",
        " ",
        teks,
        flags=re.IGNORECASE
    )


    # menghapus spasi berlebihan
    teks = re.sub(
        r"\s+",
        " ",
        teks
    )


    return teks.strip()

Fungsi bersihkan_teks(teks) digunakan untuk membersihkan teks hasil crawling dari bagian-bagian yang tidak diperlukan sebelum data digunakan untuk analisis. Pertama, program memeriksa apakah variabel teks bernilai None; jika iya, fungsi langsung mengembalikan None karena tidak ada teks yang dapat diproses. Selanjutnya, re.sub() digunakan untuk menghapus teks “ADVERTISEMENT” dan “SCROLL TO CONTINUE WITH CONTENT” yang biasanya muncul sebagai bagian dari iklan atau elemen halaman website, dengan re.IGNORECASE agar proses tetap bekerja meskipun penulisannya menggunakan huruf besar atau kecil. Setelah itu, pola \s+ digunakan untuk mengubah spasi, tab, atau baris baru yang berlebihan menjadi satu spasi sehingga teks menjadi lebih rapi. Terakhir, strip() digunakan untuk menghapus spasi yang terdapat di awal dan akhir teks. Dengan demikian, fungsi ini menghasilkan teks artikel yang lebih bersih, rapi, dan siap digunakan untuk tahap pengolahan atau analisis selanjutnya.

In [9]:
def ambil_isi_berita(url):

    try:

        response = session.get(
            url,
            timeout=25
        )

        response.raise_for_status()


        isi = trafilatura.extract(
            response.text,
            include_comments=False,
            include_tables=False,
            output_format="txt",
            favor_precision=True,
            deduplicate=True
        )


        isi = bersihkan_teks(isi)


        # menghindari halaman kosong
        # atau isi yang terlalu pendek
        if isi and len(isi) >= 300:

            return isi


    except Exception as e:

        print(
            "Gagal:",
            url
        )

        print(
            "Error:",
            e
        )


    return None

Fungsi ambil_isi_berita(url) digunakan untuk mengambil dan mengekstraksi isi utama dari sebuah halaman berita berdasarkan URL yang telah diperoleh sebelumnya. Pertama, fungsi menggunakan session.get() untuk mengakses halaman berita dengan batas waktu 25 detik, kemudian raise_for_status() digunakan untuk memastikan request berhasil dan akan menghasilkan error jika terjadi masalah pada respons. Setelah halaman berhasil diambil, trafilatura.extract() digunakan untuk mengekstraksi teks utama artikel dari HTML. Beberapa pengaturan diterapkan, yaitu include_comments=False untuk mengabaikan komentar, include_tables=False untuk tidak mengambil isi tabel, output_format="txt" untuk menghasilkan teks biasa, favor_precision=True untuk mengutamakan ketepatan dalam mengambil isi artikel, dan deduplicate=True untuk menghindari teks yang sama muncul berulang kali. Hasil ekstraksi kemudian diproses menggunakan fungsi bersihkan_teks() agar teks iklan, spasi berlebihan, dan bagian yang tidak diperlukan dapat dihapus. Selanjutnya, program memeriksa apakah teks tersedia dan memiliki panjang minimal 300 karakter. Pemeriksaan ini bertujuan untuk menghindari penyimpanan halaman yang kosong atau isi yang terlalu pendek dan tidak layak dianggap sebagai artikel. Jika memenuhi syarat, isi artikel dikembalikan melalui return isi. Namun, jika terjadi kesalahan selama proses pengambilan atau ekstraksi, program menampilkan URL dan pesan error, kemudian mengembalikan None. Dengan demikian, fungsi ini berperan dalam mengambil, mengekstraksi, membersihkan, dan memvalidasi isi berita sebelum digunakan sebagai data hasil crawling.

In [10]:
def scraping_kategori(
    links,
    label,
    target=100
):

    hasil = []


    for url in tqdm(
        links,
        desc=f"Scraping {label}"
    ):


        if len(hasil) >= target:
            break


        isi = ambil_isi_berita(url)


        if isi is not None:

            hasil.append(
                {
                    "isi_berita": isi,
                    "label": label
                }
            )


        # delay supaya tidak terlalu cepat
        time.sleep(
            random.uniform(1, 2)
        )


    print(
        f"{label} berhasil:",
        len(hasil)
    )


    return hasil

Fungsi scraping_kategori() digunakan untuk melakukan proses scraping isi berita dari sekumpulan link berdasarkan kategori tertentu sampai mencapai jumlah data yang ditargetkan. Fungsi ini menerima links sebagai daftar URL berita yang akan diproses, label sebagai penanda kategori berita, dan target=100 sebagai jumlah artikel yang ingin dikumpulkan. Program membuat list hasil untuk menyimpan artikel yang berhasil diambil. Selanjutnya, setiap URL diproses menggunakan perulangan for dan tqdm untuk menampilkan progress scraping sesuai dengan label kategori. Sebelum mengambil artikel, program memeriksa jumlah data yang sudah berhasil dikumpulkan; jika sudah mencapai target, proses dihentikan menggunakan break. Setiap URL kemudian dikirim ke fungsi ambil_isi_berita() untuk mengambil dan membersihkan isi artikel. Jika isi berita berhasil diperoleh dan tidak bernilai None, data tersebut dimasukkan ke dalam list hasil dalam bentuk dictionary yang terdiri dari isi_berita sebagai isi artikel dan label sebagai kategori berita. Setelah setiap proses pengambilan artikel, program memberikan jeda acak antara 1 hingga 2 detik menggunakan random.uniform() agar request tidak dikirim terlalu cepat secara berturut-turut. Setelah seluruh proses selesai atau target tercapai, program menampilkan jumlah artikel yang berhasil dikumpulkan dan mengembalikan list hasil. Dengan demikian, fungsi ini digunakan untuk mengubah sekumpulan kandidat URL menjadi dataset berita yang sudah berisi teks artikel dan label kategorinya.

In [11]:
data_sport = scraping_kategori(
    link_sport,
    "sport",
    target=100
)

Scraping sport:  66%|██████▌   | 103/157 [03:58<02:04,  2.31s/it]

sport berhasil: 100


Kode tersebut digunakan untuk menjalankan proses scraping terhadap link berita kategori Sport yang sebelumnya telah dikumpulkan. Fungsi scraping_kategori() dipanggil dengan link_sport sebagai daftar URL berita Sport, "sport" sebagai label kategori, dan target=100 untuk menentukan bahwa program hanya akan mengambil 100 artikel Sport yang berhasil diekstraksi. Hasil scraping kemudian disimpan dalam variabel data_sport. Setiap data yang berhasil dikumpulkan akan berisi isi berita dan label "sport", sehingga data_sport nantinya menjadi kumpulan data berita Sport yang siap digabungkan dengan kategori lainnya untuk membentuk dataset.

In [12]:
data_finance = scraping_kategori(
    link_finance,
    "finance",
    target=100
)

Scraping finance:  68%|██████▊   | 105/154 [03:42<01:43,  2.12s/it]

finance berhasil: 100


Kode tersebut digunakan untuk menjalankan proses scraping terhadap link berita kategori Finance yang sebelumnya telah dikumpulkan. Fungsi scraping_kategori() dipanggil dengan link_finance sebagai daftar URL berita Finance, "finance" sebagai label kategori, dan target=100 untuk menentukan bahwa program akan mengambil 100 artikel Finance yang berhasil diekstraksi. Hasil scraping kemudian disimpan dalam variabel data_finance. Setiap data yang berhasil dikumpulkan akan memiliki isi berita dan label "finance", sehingga data_finance nantinya dapat digabungkan dengan data_sport untuk membentuk dataset berita dari kedua kategori.

In [ ]:
data_semua = (
    data_sport
    +
    data_finance
)

Kode tersebut digunakan untuk menggabungkan data berita dari kategori Sport dan Finance menjadi satu dataset. Variabel data_sport berisi hasil scraping 100 berita kategori Sport, sedangkan data_finance berisi hasil scraping 100 berita kategori Finance. Operator + digunakan untuk menggabungkan kedua list tersebut, sehingga seluruh data disimpan dalam variabel data_semua. Dengan demikian, jika masing-masing kategori berhasil memperoleh 100 artikel, maka data_semua akan berisi 200 data berita, yaitu 100 berita Sport dan 100 berita Finance, yang masing-masing sudah memiliki isi berita dan label kategorinya.

In [14]:
df = pd.DataFrame(
    data_semua
)

Kode tersebut digunakan untuk mengubah seluruh data berita yang tersimpan dalam data_semua menjadi sebuah DataFrame menggunakan library Pandas. Fungsi pd.DataFrame() akan menyusun data yang sebelumnya berbentuk list of dictionary menjadi tabel, sehingga lebih mudah untuk dilihat, diolah, dibersihkan, maupun disimpan ke dalam file. Hasilnya disimpan dalam variabel df, yang menjadi dataset utama hasil crawling. Jika sebelumnya berhasil diperoleh 100 berita Sport dan 100 berita Finance, maka df akan berisi sekitar 200 baris data, dengan kolom seperti isi_berita dan label.

In [15]:
df.insert(
    0,
    "id",
    range(
        1,
        len(df) + 1
    )
)

Kode tersebut digunakan untuk menambahkan kolom id sebagai identitas atau nomor urut pada setiap data berita di DataFrame df. Fungsi df.insert() digunakan untuk menyisipkan kolom baru pada posisi tertentu. Angka 0 menunjukkan bahwa kolom id akan ditempatkan sebagai kolom pertama. Kemudian "id" merupakan nama kolom yang ditambahkan, sedangkan range(1, len(df) + 1) menghasilkan nomor urut yang dimulai dari 1 sampai jumlah seluruh baris pada DataFrame. Jadi, jika df memiliki 200 data berita, kolom id akan berisi angka 1 hingga 200. Penambahan ID ini memudahkan proses identifikasi, pengelolaan, dan pengecekan setiap data berita dalam dataset.

In [16]:
df = df[
    [
        "id",
        "isi_berita",
        "label"
    ]
]

Kode tersebut digunakan untuk memilih dan mengatur urutan kolom yang akan digunakan dalam DataFrame df. DataFrame kemudian hanya mempertahankan tiga kolom, yaitu id, isi_berita, dan label. Kolom id berisi nomor identitas setiap berita, isi_berita berisi teks atau isi artikel yang telah diperoleh dari proses scraping, sedangkan label menunjukkan kategori berita, yaitu sport atau finance. Dengan penulisan df = df[[...]], DataFrame lama diperbarui sehingga hanya tiga kolom tersebut yang dipertahankan dan disusun dengan urutan id, isi_berita, kemudian label. Ini membuat struktur dataset menjadi lebih sederhana dan sesuai dengan kebutuhan analisis selanjutnya.

In [17]:
df

,id,isi_berita,label
0,1,Pacuan kuda Indonesia mulai diarahkan menuju p...,sport
1,2,"Jelang Asian Games 2026, dukungan untuk 432 at...",sport
2,3,Nova Widianto mulai menangani tim ganda campur...,sport
3,4,Timnas basket putra Indonesia jadi rombongan p...,sport
4,5,"Bagi pelari urban, lari kini bukan lagi sekada...",sport
...,...,...,...
195,196,"Menteri Pertanian (Mentan), Andi Amran Sulaima...",finance
196,197,Menteri Keuangan (Menkeu) Purbaya Yudhi mengaj...,finance
197,198,Masyarakat Indonesia semakin banyak yang mengg...,finance
198,199,Presiden Prabowo Subianto meminta semua pihak ...,finance


Kode tersebut digunakan untuk menampilkan isi DataFrame df di Jupyter Notebook. Setelah kode sebelumnya, df berisi tiga kolom utama, yaitu id, isi_berita, dan label. Dengan menjalankan df, Jupyter Notebook akan menampilkan data hasil crawling dalam bentuk tabel sehingga kita bisa melihat, memeriksa, dan memastikan hasil scraping sudah tersimpan dengan benar. Jika target scraping terpenuhi, tabel tersebut akan berisi sekitar 200 baris, terdiri dari 100 berita Sport dan 100 berita Finance.

In [18]:
print(
    "Total data:",
    len(df)
)

Total data: 200


Kode tersebut digunakan untuk menampilkan jumlah keseluruhan data yang terdapat dalam DataFrame df. Fungsi len(df) menghitung jumlah baris pada DataFrame, kemudian hasilnya ditampilkan menggunakan print() dengan keterangan "Total data:". Jadi, kode ini berfungsi untuk memastikan berapa banyak berita yang berhasil dikumpulkan dari proses crawling. Jika scraping berhasil mendapatkan 100 berita Sport dan 100 berita Finance, maka hasilnya akan menampilkan Total data: 200.

In [19]:
print(
    df["label"].value_counts()
)

label
sport      100
finance    100
Name: count, dtype: int64


Kode tersebut digunakan untuk menghitung jumlah data pada setiap kategori berita yang terdapat dalam kolom label pada DataFrame df. df["label"] digunakan untuk mengambil kolom label, kemudian value_counts() menghitung berapa kali setiap kategori muncul. Hasilnya ditampilkan menggunakan print(). Dalam dataset ini, kode tersebut digunakan untuk memastikan keseimbangan jumlah data antara kategori Sport dan Finance.

In [20]:
nama_file = "dataset_detik_200_berita.xlsx"


df.to_excel(
    nama_file,
    index=False
)


print(
    "File berhasil disimpan:",
    nama_file
)

File berhasil disimpan: dataset_detik_200_berita.xlsx


Kode tersebut digunakan untuk menyimpan DataFrame df ke dalam file Excel agar hasil crawling dapat digunakan atau dianalisis kembali. Variabel nama_file menentukan nama file yang akan dibuat, yaitu dataset_detik_200_berita.xlsx. Selanjutnya, df.to_excel() digunakan untuk mengubah DataFrame menjadi file Excel, sedangkan index=False digunakan agar nomor indeks bawaan DataFrame tidak ikut disimpan sebagai kolom tambahan. Setelah proses penyimpanan selesai, print() menampilkan pesan bahwa file telah berhasil disimpan beserta nama filenya. Jadi, kode ini berfungsi untuk menyimpan dataset hasil crawling yang berisi sekitar 200 berita Sport dan Finance ke dalam format Excel.

In [21]:
import os

print(os.path.abspath("dataset_detik_200_berita.xlsx"))

C:\Users\ASUS\PPW\dataset_detik_200_berita.xlsx


Kode tersebut digunakan untuk mengetahui lokasi atau path lengkap tempat file Excel dataset_detik_200_berita.xlsx disimpan. Library os diimpor untuk menyediakan fungsi-fungsi yang berkaitan dengan sistem operasi. Kemudian os.path.abspath() mengubah nama file relatif menjadi alamat atau lokasi file secara lengkap berdasarkan folder kerja Python saat ini. Hasilnya ditampilkan menggunakan print(), sehingga kita dapat mengetahui secara pasti di folder mana file dataset hasil crawling tersimpan. Dengan demikian, kode ini berguna untuk mengecek lokasi file Excel setelah proses penyimpanan selesai, terutama jika kita ingin membuka atau memindahkan file tersebut.